In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from numpy.random import random_sample,randn
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic

g = random_sample(int(1e5))*10 # uniform random values between 0 and 10
p = abs(randn(int(1e5))) # abs of normally distributed data

"""
plot g vs p in groups with different colors
colors are cycled automatically by matplotlib
use another colormap or define own colors for a different cycle
"""
for i in range(1,11): 
    plt.plot(g[abs(g-i)<1], p[abs(g-i)<1], ls='', marker='.')

plt.show()

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])


In [ ]:
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6


In [ ]:
X = np.hstack((power,coherence,granger))
X = X[indx_tot]

print(mouse.shape)
print(X.shape)

X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]

mu = 1.0

#>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# Add the newer data
power,coherence,granger,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X_new = np.hstack((power,coherence,granger))

windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])


In [ ]:
labels_new.keys()

In [ ]:
type(labels_new['gcFeatures'])

In [ ]:
labels_new['gcFeatures'].shape

In [ ]:
pf = labels_new['powerFeatures']
cf = labels_new['cohFeatures']
gf = labels_new['gcFeatures']
ft = np.concatenate((pf,cf,gf))

In [ ]:
ft.shape

In [ ]:
ft = np.concatenate(np.squeeze(labels_new['powerFeatures']),np.squeeze(labels_new['cohFeatures']),
                    np.squeeze(labels_new['gcFeatures']))

In [ ]:
idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]


ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1

print('>>>>>>>>>>>>>.')
print(y_new.shape)
print(ids.shape)
print(mouse_new.shape)
print(X_new.shape)

X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]


In [ ]:
X_train_tot = np.vstack((X_train,X_train_new))
y_train_tot = np.concatenate((y_train,y_train_new))
weights_g = np.ones(X_train_tot.shape[0])
weights_s = np.ones(X_train_tot.shape[0])

nFact = 8
myDict = {}


In [ ]:
np.mean(y_train_tot)

In [ ]:
from scipy.stats import ranksums

In [ ]:
ss = np.zeros(X_train_tot.shape[1])
pp = np.zeros(X_train_tot.shape[1])
for i in range(X_train_tot.shape[1]):
    ss[i],pp[i] = ranksums(X_train_tot[y_train_tot==1,i],X_train_tot[y_train_tot==0,i])

In [ ]:
sign = np.sign(np.mean(X_train_tot[y_train_tot==1],axis=0)-np.mean(X_train_tot[y_train_tot==0],axis=0))
mean_diff = np.mean(X_train_tot[y_train_tot==1],axis=0)-np.mean(X_train_tot[y_train_tot==0],axis=0)

In [ ]:
ii = np.arange(X_train_tot.shape[1])
ii.shape

In [ ]:
plt.scatter(ii,-np.log(pp),s=.1)

In [ ]:
9856/11

In [ ]:
plt.scatter(ii[:56*11],-np.log(pp[:56*11]),s=.1)
plt.title('Power Features')

In [ ]:
pf[::56]

In [ ]:
sval = 3.0
plt.scatter(ii[:56],-np.log(pp[:56]),s=sval,label='IL',
            c='rosybrown')
plt.scatter(ii[56*1:56*2],-np.log(pp[56*1:56*2]),s=sval,
            label='LHb',c='teal')
plt.scatter(ii[56*2:56*3],-np.log(pp[56*2:56*3]),s=sval,
            label='LSN',c='olivedrab')
plt.scatter(ii[56*3:56*4],-np.log(pp[56*3:56*4]),s=sval,
            label='MDThal',c='aqua')
plt.scatter(ii[56*4:56*5],-np.log(pp[56*4:56*5]),s=sval,
            label='MeA',c='fuchsia')
plt.scatter(ii[56*5:56*6],-np.log(pp[56*5:56*6]),s=sval,
            label='NAc',c='maroon')
plt.scatter(ii[56*6:56*7],-np.log(pp[56*6:56*7]),s=sval,
            label='OFC',c='khaki')
plt.scatter(ii[56*7:56*8],-np.log(pp[56*7:56*8]),s=sval,
            label='PL',c='darkslategrey')
plt.scatter(ii[56*8:56*9],-np.log(pp[56*8:56*9]),s=sval,
            label='V1',c='plum')
plt.scatter(ii[56*9:56*10],-np.log(pp[56*9:56*10]),s=sval,
            label='VHipp',c='deeppink')
plt.scatter(ii[56*10:56*11],-np.log(pp[56*10:56*11]),s=sval,
            label='VMHvl',c='red')
plt.ylabel('Log Pvalue',fontsize=16)
plt.title('Power Features',fontsize=20)
plt.legend(loc='upper center')
plt.savefig('PowerFeaturesManhattan.png',dpi=300)

In [ ]:
myDict = {}
myDict['pvalues'] = pp
myDict['statistics'] = ss
myDict['features'] = ft
from scipy.io import savemat
savemat('ManhattanPlot.mat',myDict)

In [ ]:
myDict2 = {}
myDict2['sign'] = sign
myDict2['mean_diff'] = mean_diff
savemat('SignManhattan.mat',myDict2)

In [ ]:
sign.shape

In [ ]:
mean_diff.shape

In [ ]:
sval = 3.0
plt.scatter(ii[:56],-np.log(pp[:56])*sign[:56],s=sval,label='IL',
            c='rosybrown')
plt.scatter(ii[56*1:56*2],-np.log(pp[56*1:56*2])*sign[56*1:56*2],s=sval,
            label='LHb',c='teal')
plt.scatter(ii[56*2:56*3],-np.log(pp[56*2:56*3])*sign[56*2:56*3],s=sval,
            label='LSN',c='olivedrab')
plt.scatter(ii[56*3:56*4],-np.log(pp[56*3:56*4])*sign[56*3:56*4],s=sval,
            label='MDThal',c='aqua')
plt.scatter(ii[56*4:56*5],-np.log(pp[56*4:56*5])*sign[56*4:56*5],s=sval,
            label='MeA',c='fuchsia')
plt.scatter(ii[56*5:56*6],-np.log(pp[56*5:56*6])*sign[56*5:56*6],s=sval,
            label='NAc',c='maroon')
plt.scatter(ii[56*6:56*7],-np.log(pp[56*6:56*7])*sign[56*6:56*7],s=sval,
            label='OFC',c='khaki')
plt.scatter(ii[56*7:56*8],-np.log(pp[56*7:56*8])*sign[56*7:56*8],s=sval,
            label='PL',c='darkslategrey')
plt.scatter(ii[56*8:56*9],-np.log(pp[56*8:56*9])*sign[56*8:56*9],s=sval,
            label='V1',c='plum')
plt.scatter(ii[56*9:56*10],-np.log(pp[56*9:56*10])*sign[56*9:56*10],s=sval,
            label='VHipp',c='deeppink')
plt.scatter(ii[56*10:56*11],-np.log(pp[56*10:56*11])*sign[56*10:56*11],s=sval,
            label='VMHvl',c='red')
plt.ylabel('Log Pvalue',fontsize=16)
plt.title('Power Features',fontsize=20)
plt.legend(loc='upper center')
plt.savefig('PowerFeaturesManhattan_2.png',dpi=300)

In [ ]:
plt.plot(sign[:56*11])

In [ ]:
plt.plot(mean_diff[:56*11])